# Data Quality Report

Flags implausible delays, stale observations, pre-trip/post-trip observations, placeholder zero delays (buses waiting at the first stop, trips without realtime data), and VM vs stop-call delay disagreement before delay metrics are computed.

Results come from the shared DuckDB cache under `outputs/report-cache/`, the same path the CLI scripts use by default. The first run builds it (about 1.5 minutes, DuckDB capped at 3 GB); later runs reuse it. The legacy `load_observations` path loads every observation into pandas and needs more memory than a 15 GiB machine has for the full database.

In [ ]:
from pathlib import Path
import importlib.util
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "analysis"))

spec = importlib.util.spec_from_file_location(
    "data_quality_report",
    PROJECT_ROOT / "analysis" / "data-quality-report.py",
)
quality = importlib.util.module_from_spec(spec)
spec.loader.exec_module(quality)

DB = PROJECT_ROOT / "data" / "foli.db"
LIMIT = 20
MIN_OBSERVATIONS = 50

CACHE_DIR = PROJECT_ROOT / "outputs" / "report-cache"
MIN_TRIPS = 10

import cached_queries

In [ ]:
class Args:
    db = DB
    cache_dir = CACHE_DIR
    min_observations = MIN_OBSERVATIONS
    limit = LIMIT
    view = "summary"

cached_queries.quality_report(Args)

In [ ]:
Args.view = "line"
cached_queries.quality_report(Args)

In [ ]:
Args.view = "examples"
cached_queries.quality_report(Args)